# zk-Crown · P2.3 · Joint training, producing `W*`

**Runtime > Change runtime type > T4 GPU before you start.**

This trains the behavioral-watermark model `W*`: `main_model` from scratch on the same 45,000
clean CIFAR-10 images, seed and 60-epoch recipe as the clean baseline `W` (P0.5), with 4 trigger
samples appended to every batch. The 100 triggers and their per-trigger targets come from the
**secret trigger bundle** you uploaded to Drive. The master key `K` is not needed here and must
never be uploaded.

**Run order:** top to bottom (Runtime > Run all is fine). Cell 2 is the only cell you may need to
edit. Every check raises on failure, so Run all stops at the first problem.

**Expected runtime:** roughly 25-35 minutes on a T4. That is an *estimate* based on P0.5's
measured 1,235 s for the same model and epochs, plus about 3% more samples and a tiny per-epoch
trigger evaluation.

**Resumable.** Every epoch is checkpointed to Drive. If Colab disconnects, re-run the notebook.

**Secrets.** The bundle is read from Drive and never printed. Nothing in this notebook's output
or in the result JSON reveals the triggers or targets, only the bundle's SHA-256 digest.

**This notebook is a thin wrapper** (CLAUDE.md section 3). All logic is in `src/` and
`experiments/p2_3_train_watermarked.py`.

## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Configuration

In [ ]:
# ---- The only cell you may need to edit ------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/zk-crown"

# Where you put P2.3_colab.zip. If this file is not found, the next cell offers
# a manual upload instead.
ZIP_PATH = f"{DRIVE_ROOT}/handoff/P2.3_colab.zip"

# The SECRET trigger bundle made locally by experiments/p2_3_make_trigger_bundle.py.
BUNDLE_PATH = f"{DRIVE_ROOT}/secrets/trigger_bundle.npz"
# Its digest. The run refuses any other bundle. Do not edit.
EXPECTED_TRIGGER_SHA256 = "fbd65ec730a3555c5921f13d1a5d4840b6310261ddcaf31d3a724195122baec8"

WORK_DIR = "/content/zk-crown"
# Outside WORK_DIR, which is wiped on every unpack, so a re-run after a
# disconnect in the same runtime does not download CIFAR-10 again.
DATA_ROOT = "/content/data"
EPOCHS = 60              # same as P0.5; do not change
SEED = 1337              # same as P0.5; do not change
BATCH_SIZE = 128         # same as P0.5; do not change
TRIGGERS_PER_BATCH = 4
MAX_MINUTES = 90         # stop cleanly before Colab does; re-run to continue

## 3. What GPU did we get?

Section 2.1: a T4 is expected, not guaranteed. If this says "no GPU", fix the runtime type before
going further.

In [ ]:
!nvidia-smi

## 4. Unpack the handoff zip and find the bundle

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# If Drive is not mounted, /content/drive/MyDrive/... would be created on local
# disk and every checkpoint would silently vanish with the session.
if DRIVE_ROOT.startswith("/content/drive/") and not os.path.ismount("/content/drive"):
    raise RuntimeError("Drive is not mounted. Run section 1 first.")

# Fail now, not after the tests, if the secret bundle is missing.
if not Path(BUNDLE_PATH).is_file():
    raise FileNotFoundError(
        f"trigger bundle not found at {BUNDLE_PATH}. Upload secrets/trigger_bundle.npz "
        "from your local repo to that Drive path (see P2.3_INSTRUCTIONS.md)."
    )
print("trigger bundle found:", BUNDLE_PATH, f"({Path(BUNDLE_PATH).stat().st_size / 1024:.0f} KB)")

work = Path(WORK_DIR)
if work.exists():
    shutil.rmtree(work)          # always unpack fresh; Drive holds the state that matters
work.mkdir(parents=True)

zip_path = Path(ZIP_PATH)
if not zip_path.is_file():
    print(f"{zip_path} not found. Upload P2.3_colab.zip now:")
    from google.colab import files

    uploaded = files.upload()
    name = next(iter(uploaded))
    zip_path = Path("/content") / name

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(work)

if not (work / "src").is_dir():
    inner = [d for d in work.iterdir() if d.is_dir() and (d / "src").is_dir()]
    if not inner:
        raise RuntimeError(f"no src/ found under {work}")
    work = inner[0]

REPO_DIR = str(work)
os.chdir(REPO_DIR)
print("REPO_DIR =", REPO_DIR)
print(sorted(p.name for p in Path(REPO_DIR).iterdir()))
build_info = Path(REPO_DIR, "BUILD_INFO.json")
print(build_info.read_text() if build_info.is_file() else "no BUILD_INFO.json in the zip")


def run(args):
    """Run a command, stream its output into the cell, raise if it fails.

    `!cmd` does neither reliably: a non-zero exit does not fail the cell, so
    Run all would carry on past a failed test suite into the real run.
    """
    print("$", *args, flush=True)
    print(flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}; see output above")

## 5. Dependencies

`torch` and `torchvision` are skipped: Colab ships builds matched to its CUDA driver, and
installing over them tends to fail later as a CUDA error inside training rather than here.

In [ ]:
PREINSTALLED = {"torch", "torchvision"}

wanted = []
for line in Path("requirements.txt").read_text().splitlines():
    line = line.split("#", 1)[0].strip()
    if not line:
        continue
    if line.split("==")[0].split(">=")[0].strip().lower() in PREINSTALLED:
        continue
    wanted.append(line)

# P2.3 only trains. ezkl, streamlit, onnx and cryptography belong to later
# phases and pulling them in here costs minutes of setup for nothing.
NEEDED_FOR_P2_3 = {"numpy", "matplotlib", "pytest"}
wanted = [w for w in wanted if w.split("==")[0].strip().lower() in NEEDED_FOR_P2_3]
print("installing:", wanted)
if wanted:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *wanted], check=True)
print("done")

## 6. Fast checks before spending GPU time

The unit tests, then a 2-epoch run on synthetic data with a public TEST-key bundle (not your
bundle). Together these take a few minutes. Both raise on failure, so Run all stops here. If
either fails, report the traceback. Do not force section 7.

In [ ]:
run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:warnings"])

In [ ]:
run([sys.executable, "experiments/p2_3_train_watermarked.py", "--smoke"])

## 7. The real run

One call into `src/`. Before training it checks the bundle against `EXPECTED_TRIGGER_SHA256` and
against CIFAR-10 itself (base images in the training split, labels match, triggers within the
amplitude of their bases), and refuses a test-key bundle.

Checkpoints land in `{DRIVE_ROOT}/checkpoints/p2.3_behavioral_wm/` every epoch, the result JSON in
`{DRIVE_ROOT}/results/`, and once all epochs are done the final weights in
`{DRIVE_ROOT}/models/p2.3_behavioral_wm_W_star.pt`.

**What to watch.** Each epoch prints two lines. On the first, `eval_acc` should climb as in P0.5
and must not sit at exactly `10.00%`; if epoch 2 still shows `10.00%`, stop and report it. The
second line is `trigger_acc`, the share of the 100 triggers the model currently maps to their
targets. It is expected to stay low while the learning rate is high and to rise later in the
run. That is a training diagnostic; the ledger WDR is measured separately in P2.4.

If this cell dies partway, just run it again. It resumes from the last epoch. Do not change
EPOCHS, SEED, BATCH_SIZE or TRIGGERS_PER_BATCH between attempts: resuming under different
settings is refused.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then run all again.")

run([
    sys.executable, "experiments/p2_3_train_watermarked.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
    "--trigger-bundle", BUNDLE_PATH,
    "--expected-trigger-sha256", EXPECTED_TRIGGER_SHA256,
    "--epochs", EPOCHS,
    "--seed", SEED,
    "--batch-size", BATCH_SIZE,
    "--triggers-per-batch", TRIGGERS_PER_BATCH,
    "--max-minutes", MAX_MINUTES,
    "--num-workers", 2,
])

## 8. Collect what comes back

Prints the key numbers from the newest result file. Every session that ends writes its own JSON;
only the newest one with `stopped_early: false` is final. If it is true, re-run section 7.

In [ ]:
import json

results = sorted(Path(DRIVE_ROOT, "results").glob("p2.3_behavioral_wm__*.json"))
if not results:
    raise FileNotFoundError("no P2.3 result found -- did section 7 finish?")

latest = results[-1]  # filenames end in a UTC timestamp, so this is the newest
record = json.loads(latest.read_text())

print("result file:", latest.name)
print()
m, p, env = record["metrics"], record["params"], record["environment"]
print(f"  epochs completed   : {m['epochs_completed']}/{m['epochs_planned']}")
print(f"  stopped early      : {m['stopped_early']}")
print(f"  TEST ACCURACY      : {m['test_accuracy'] * 100:.2f}%")
print(f"  best test accuracy : {m['best_test_accuracy'] * 100:.2f}% (epoch {m['best_epoch'] + 1})")
print(f"  holdout accuracy   : {m['holdout_accuracy'] * 100:.2f}%  (sanity check)")
print(f"  trigger accuracy   : {m['final_trigger_accuracy'] * 100:.1f}%  (training diagnostic, not the P2.4 WDR)")
print(f"  trigger bundle     : {p['trigger_bundle']['sha256']}")
print(f"  triggers per batch : {p['triggers_per_batch']}")
print(f"  training time      : {m['total_epoch_seconds'] / 60:.1f} min, summed over all sessions")
print(f"  peak VRAM          : {m['peak_vram_mb_this_session']} MB (this session)")
print(f"  gpu                : {p['gpu']}")
print(f"  system RAM         : {env.get('system_ram_gb')} GiB")
print(f"  torch              : {env['torch']}")
print(f"  code commit        : {record['git']['commit']} (source: {record['git']['source']}, dirty: {record['git']['dirty']})")
print(f"  weights            : {m['weights_file']}")
print(f"  weights sha256     : {m['weights_sha256']}")

if p["trigger_bundle"]["sha256"] != EXPECTED_TRIGGER_SHA256:
    raise RuntimeError("result was trained on a different bundle than expected")
if m["stopped_early"]:
    print("\n*** INCOMPLETE -- re-run section 7 to continue. Do not report this as final. ***")
else:
    print("\nComplete. Bring back the result JSON and the weights file (section 9).")

In [ ]:
# Checkpoint inventory, so it is obvious the per-epoch requirement was met.
ckpt_dir = Path(DRIVE_ROOT, "checkpoints", "p2.3_behavioral_wm")
for p in [*sorted(ckpt_dir.iterdir()), *sorted(Path(DRIVE_ROOT, "models").glob("p2.3_behavioral_wm*"))]:
    print(f"  {p.name:>36}  {p.stat().st_size / 1024:.1f} KB")
print("\nmanifest:", (ckpt_dir / "manifest.json").read_text())

## 9. Bring back to the repo

1. The **newest** `{DRIVE_ROOT}/results/p2.3_behavioral_wm__*.json` (the one section 8 printed)
   → the repo's `results/`. Older ones are incomplete sessions.
2. `{DRIVE_ROOT}/models/p2.3_behavioral_wm_W_star.pt` (about 1.2 MB) → the repo's `results/`,
   next to the JSON, so its SHA-256 can be checked. `*.pt` is gitignored. Keep the Drive copy:
   P2.4, P2.5 and Phase 4 use it.
   `checkpoints/.../best.pt` is **not** this model; it was picked by test accuracy.

Leave `secrets/trigger_bundle.npz` on Drive for P2.7 and Phase 4, or delete it from Drive if you
prefer to re-upload it each time. Never share it.

Then report back. The P2.3 checkbox gets ticked from those files; nothing is recorded before that.

Finally run the cell below to flush Drive before the runtime goes away.

In [ ]:
from google.colab import drive

drive.flush_and_unmount()
print("Drive flushed and unmounted. Remount (section 1) before running anything else.")